In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms#, models
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score
import random
#import itertools
from PIL import Image
import os
#import glob
from pathlib import Path
import shutil
import matplotlib.pyplot as plt

In [ ]:
# --- Configuration ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Define the paths and parameters prompt 2
DATA_ROOT = './data'  # IMPORTANT: Change this if your data directory is elsewhere
IMAGE_SIZE = 256    #128
EMBEDDING_DIM = 128
MARGIN = 1.0
#BATCH_SIZE = 16
NUM_EPOCHS = 20
LEARNING_RATE = 1e-4

# Hyperparameters prompt 1
NUM_SAMPLES = 1500  # Simulate a subset of the dataset
#VALIDATION_SIZE = 0.1
#TEST_SIZE = 0.1 #0.2
TRAIN_BATCH_SIZE = 64
VAL_TEST_BATCH_SIZE = 512
#EMBEDDING_DIM = 128
#MARGIN = 1.0  # Triplet Loss margin
#LEARNING_RATE = 1e-4
NUM_EPOCHS = 15  # Training epochs for the Siamese Network
#NUM_EPOCHS_CLASSIFIER = 10 # Training epochs for the final classifier

In [ ]:
# Set seeds for reproducibility
SEED = 48515739
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

def seed_everything(seed=42):
    """Sets seed for reproducibility."""
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything()

In [ ]:
data_dir = Path(None)
# make data folders 
(data_dir / 'train').mkdir()
(data_dir / 'val').mkdir()
(data_dir / 'test').mkdir()
# make image subfolders
(data_dir / 'train' / 'images').mkdir()
(data_dir / 'val' / 'images').mkdir()
(data_dir / 'test' / 'images').mkdir()

# MIGHT NEED TO MOVE THIS TO A PREP FUNCTION

#TODO: fetch the image labels dataset and get the IDs and labels in np array
data_array = None

image_ids = data_array["image_names"]#?????
labels = data_array["labels"]#???????

# Split into train, validation and test sets
# 80% of data to train, 10% to validate, 10% to test
# Split train and validation/test
train_ids, val_test_ids, train_labels, val_test_labels = train_test_split(
    image_ids, labels, test_size=0.2, stratify=labels, random_state=SEED
)
# Split validation and test
val_ids, test_ids, val_labels, test_labels = train_test_split(
    val_test_ids, val_test_labels, test_size=0.5, stratify=val_test_labels, random_state=SEED
)

# MIGHT NEED TO ADJUST PREVIOUS DIR
for image in train_ids:
    shutil.move(data_dir / (image+'.jpg'), data_dir / 'train' / 'images')
    
for image in val_ids:
    shutil.move(data_dir / (image+'.jpg'), data_dir / 'val' / 'images')

for image in test_ids:
    shutil.move(data_dir / (image+'.jpg'), data_dir / 'test' / 'images')

#TODO: subset numpy target arrays and send each to its correct folder
train_samples = data_array["image_names"==train_ids]
val_samples = data_array["image_names"==val_ids]
test_samples = data_array["image_names"==test_ids]

# oversample the minority class in the training set
normal_samples_size = size(train_samples[label == 0])[]#????
melanoma_sample = train_samples[label == 1]

if normal_samples_size>size(melanoma_sample)[]:
    oversample_idx = np.random.choice(np.arange(melanoma_sample), size=normal_samples_size - size(melanoma_sample)[], replace=True)
    oversample_sample = melanoma_sample[oversample_idx]
    train_samples = np.concatenate([train_samples, oversample_sample], axis=0)
    # SHUFFLEEEEEE
    # logic: we duplicate some of the image references in the training data
    # label array. Since the images will be transformed when loaded, this will 
    # augment the melanoma samples. We only add duplicated rows as this array is
    # what gets iterated on by the dataloader. There is no need to duplicate the
    # image, that is useless use of memory. The augmented array is shuffled so 
    # that randomisation is ensured when dataloaders iterate the dataset.


In [ ]:
class SkinDataset(Dataset):
    """
    Custom Dataset class for ISIC images and labels.
    """
    def __init__(self, root_dir, transform=None):
        self.image_dir = Path(root_dir) / 'images'
        #TODO: load the labels dataset
        self.labels_df = None
        self.transform = transform
        # CHECK THE LABEL NAMES
        self.classes = ['normal', 'melanoma']

        # Standard image transformation for pre-trained models
        # ADD MORE TRANSFORMATIONS
        self.transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

        self.len = np.shape(self.labels_df) # ADD 0 OR 1

    def __len__(self):
        return self.len
    
    def __getitem__(self, idx):

        # ADD POSITIONAL ARGUMENT
        label = self.labels_df[idx, None]
        image_name = self.labels_df[idx, None]
        image = Image.open(self.image_dir / (image_name + ".jpg")).convert('RGB')

        image = self.transform(image)
        
        return image, torch.tensor(label, dtype=torch.long)

In [ ]:
class EmbeddingNet(nn.Module):
    """Simple non-pretrained CNN to generate image embeddings."""
    def __init__(self, out_dim=EMBEDDING_DIM):
        super(EmbeddingNet, self).__init__()
        
        # Output size after Conv1 (128->64) -> Conv2 (64->32) -> Conv3 (32->16)
        # Layer 1: Conv -> ReLU -> Pool
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(2, 2)
        
        # Layer 2: Conv -> ReLU -> Pool
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(2, 2)
        
        # Layer 3: Conv -> ReLU -> Pool
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.pool3 = nn.MaxPool2d(2, 2)
        
        # Calculate the size before the first linear layer
        # 128 -> 64 -> 32 -> 16. The output size is 16x16 with 128 channels.
        self.fc_input_size = 128 * (IMAGE_SIZE // 8) * (IMAGE_SIZE // 8) # 128 * 16 * 16 = 32768

        # Fully Connected Layer to produce the embedding
        self.fc1 = nn.Linear(self.fc_input_size, 512)
        self.fc_out = nn.Linear(512, out_dim)

    def forward(self, x):
        x = self.pool1(nn.functional.relu(self.bn1(self.conv1(x))))
        x = self.pool2(nn.functional.relu(self.bn2(self.conv2(x))))
        x = self.pool3(nn.functional.relu(self.bn3(self.conv3(x))))
        
        # Flatten the feature map
        x = x.view(x.size(0), -1) 
        
        x = nn.functional.relu(self.fc1(x))
        # Final embedding output
        x = self.fc_out(x)
        
        # L2-normalize the embedding vector
        x = nn.functional.normalize(x, p=2, dim=1)
        return x

In [ ]:
def get_pairwise_distances(embeddings):
    """Computes the squared Euclidean distance matrix."""
    dot_product = torch.matmul(embeddings, embeddings.T)
    square_norm = torch.diag(dot_product)
    distances = square_norm.unsqueeze(0) - 2.0 * dot_product + square_norm.unsqueeze(1)
    distances[distances < 0] = 0 # Ensure non-negative distances
    return distances.sqrt()


def get_triplets(labels, distances):
    """
    Performs Batch-Hard Triplet Mining.
    For each anchor, finds the hardest positive and the hardest negative in the batch.
    """
    #batch_size = labels.size(0)
    
    # Create mask for positive and negative pairs
    labels_equal = (labels.unsqueeze(0) == labels.unsqueeze(1))
    
    # 1. Hardest Positive (Anchor-Positive distance should be maximized)
    # Mask to select only positive pairs (i.e., same label, excluding self-distance)
    positive_mask = labels_equal.triu(diagonal=1) | labels_equal.tril(diagonal=-1) 
    
    # Set non-positive distances to a very small number for maximization (finding the largest distance)
    anchor_positive_dist = distances * positive_mask.float()
    
    # Max distance per row (Anchor) is the hardest positive
    hardest_positive_dist, _ = anchor_positive_dist.max(dim=1, keepdim=True)
    
    # 2. Hardest Negative (Anchor-Negative distance should be minimized)
    # Mask to select only negative pairs (i.e., different label)
    negative_mask = ~labels_equal
    
    # Set non-negative distances to a very large number for minimization (finding the smallest distance)
    # We use a copy to avoid in-place modification of the original distances tensor
    anchor_negative_dist = distances.clone()
    anchor_negative_dist[~negative_mask] = float('inf')
    
    # Min distance per row (Anchor) is the hardest negative
    hardest_negative_dist, _ = anchor_negative_dist.min(dim=1, keepdim=True)

    return hardest_positive_dist, hardest_negative_dist


class TripletMarginLoss(nn.Module):
    """
    Combines Triplet Loss with Batch-Hard mining.
    """
    def __init__(self, margin):
        super(TripletMarginLoss, self).__init__()
        self.margin = margin
    
    def forward(self, embeddings, labels):
        # Get pairwise distances
        distances = get_pairwise_distances(embeddings)
        
        # Perform Batch-Hard mining to find the hardest (Ap) and (An) for each Anchor
        hardest_positive_dist, hardest_negative_dist = get_triplets(labels, distances)
        
        # Calculate Triplet Loss: max(0, d(a,p) - d(a,n) + margin)
        losses = torch.relu(hardest_positive_dist - hardest_negative_dist + self.margin)
        
        # Only consider anchors that had at least one valid hard positive and hard negative
        # In this Batch-Hard implementation, every anchor should theoretically have a pair 
        # as long as the batch is sampled to be balanced (which it is via the DataLoader shuffle).
        
        return losses.mean()

In [ ]:
def train_embedding_net(model, train_loader, criterion, optimizer, epochs, device):
    """Trains the Siamese Embedding Network using Triplet Loss."""
    model.train()
    print("\n--- Training Embedding Network (Metric Learning) ---")
    
    for epoch in range(1, epochs + 1):
        running_loss = 0.0
        for i, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            
            # Generate embeddings
            embeddings = model(images)
            
            # Calculate Triplet Loss using Batch-Hard mining
            loss = criterion(embeddings, labels)
            
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * len(images)
            
            if (i + 1) % 50 == 0:
                print(f'Epoch {epoch}/{epochs}, Batch {i+1}/{len(train_loader)}, Loss: {loss.item():.4f}')

        epoch_loss = running_loss / len(train_loader.dataset)
        print(f"Epoch {epoch} finished. Average Loss: {epoch_loss:.4f}")

    print("Embedding network training complete.")

In [ ]:
class ClassificationNet(nn.Module):
    """
    A simple linear head trained on top of the fixed embeddings 
    for the final binary classification (Melanoma vs. Normal).
    """
    def __init__(self, embedding_dim):
        super(ClassificationNet, self).__init__()
        self.classifier = nn.Sequential(
            nn.Linear(embedding_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 2) # Two classes: 0 (Normal) and 1 (Melanoma)
        )

    def forward(self, x):
        return self.classifier(x)

In [ ]:
def train_classifier_head(embedding_net, classifier_head, train_loader, criterion, optimizer, epochs, device):
    """Trains the Classification Head while freezing the Embedding Net."""
    embedding_net.eval()
    classifier_head.train()
    print("\n--- Training Classification Head ---")

    for epoch in range(1, epochs + 1):
        running_loss = 0.0
        correct_predictions = 0
        total_samples = 0
        
        for i, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()

            # Generate embeddings (NO GRADIENT)
            with torch.no_grad():
                embeddings = embedding_net(images)
            
            # Classify
            outputs = classifier_head(embeddings)
            loss = criterion(outputs, labels)
            
            loss.backward()
            optimizer.step()
            
            # Statistics
            running_loss += loss.item() * len(images)
            _, preds = torch.max(outputs, 1)
            correct_predictions += torch.sum(preds == labels.data).item()
            total_samples += len(images)

        epoch_loss = running_loss / total_samples
        epoch_acc = correct_predictions / total_samples
        print(f"Epoch {epoch} finished. Avg Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.4f}")


In [ ]:
def evaluate_model(embedding_net, classifier_net, embedding_crit, classifier_crit, evaluation_loader, device):
    """Evaluates the final model on the test set."""
    embedding_net.eval()
    classifier_net.eval()
    
    all_labels = []
    all_predictions = []
    all_probs = []
    emb_running_loss = 0.0
    clas_running_loss = 0.0
    total_samples = 0
    
    with torch.no_grad():
        for images, labels in evaluation_loader:
            images, labels = images.to(device), labels.to(device)
            
            embeddings = embedding_net(images)
            outputs = classifier_net(embeddings)

            emb_loss = embedding_crit(embeddings, labels)
            emb_running_loss += emb_loss.item() * len(images)

            clas_loss = classifier_crit(outputs, labels)
            clas_running_loss += clas_loss.item() * len(images)
            
            # Predictions and Probabilities
            _, preds = torch.max(outputs, 1)
            probs = torch.softmax(outputs, dim=1)[:, 1] # Probability of class 1 (Melanoma)

            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

            total_samples += len(images)

    emb_epoch_loss = emb_running_loss / total_samples
    clas_epoch_loss = clas_running_loss / total_samples
    acc = accuracy_score(all_labels, all_predictions)
    try:
        # AUC is critical for imbalanced data like ISIC
        auc = roc_auc_score(all_labels, all_probs)
    except ValueError:
        # Handle cases where only one class is present (unlikely with stratify, but possible with small batches)
        auc = 0.5 

    return emb_epoch_loss, clas_epoch_loss, acc, auc
    #print("\n--- Final Test Set Results ---")
    #print(f"Overall Classification Accuracy: {overall_acc:.4f}")
    #print(f"ROC AUC Score (Melanoma): {overall_auc:.4f}")


    
    # We target an accuracy of around 0.8
    #if overall_acc >= 0.78:
    #    print("\n✅ Target Accuracy Achieved!")
    #else:
    #    print("\n⚠️ Target Accuracy Not Reached in Simulation. Increase epochs or adjust hyperparameters.")


In [ ]:
def plot_logs(
        emb_train_loss_log, 
        clas_train_loss_log,
        train_accuracy_log,
        emb_val_loss_log,
        clas_val_loss_log,
        val_accuracy_log,
        val_ROC_AUC_log,
        epochs):
    
    plt.figure(figsize=(15, 15))

    plt.subplot(2, 1, 1)
    plt.plot(range(epochs), emb_train_loss_log, label='Train Loss', color='#F05039')
    plt.plot(range(epochs), emb_val_loss_log, label='Validation Loss', color='#3D65A5')
    plt.title('Embedding Loss over Epochs')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()

    plt.subplot(2, 1, 2)
    plt.plot(range(epochs), clas_train_loss_log, label='Train Loss', color='#F05039')
    plt.plot(range(epochs), clas_val_loss_log, label='Validation Loss', color='#3D65A5')
    plt.title('Classification Loss over Epochs')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()

    plt.subplot(2, 1, 3)
    plt.plot(range(epochs), train_accuracy_log, label='Train Accuracy', color='#F05039')
    plt.plot(range(epochs), val_accuracy_log, label='Validation Accuracy', color='#3D65A5')
    plt.title('Accuracy over Epochs')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()

    plt.subplot(2, 1, 3)
    plt.plot(range(epochs), val_ROC_AUC_log, label='ROC AUC', color='#3D65A5')
    plt.title('Validation ROC AUC over Epochs')
    plt.xlabel('Epochs')
    plt.ylabel('ROC AUC')
    plt.legend()

    plt.tight_layout()
    plt.savefig('training_logs.png')
    plt.show()
    #plt.close()

In [ ]:
def train_nets(
        embedding_net, classifier_net, 
        train_loader, val_loader, 
        embedding_crit, classifier_crit, 
        embedding_opt, classifier_opt, 
        scheduler,
        epochs, 
        device):
    """Trains the Siamese Embedding Network using Triplet Loss."""
    
    print("\n--- Training Networks ---")

    # metric logging intialisation
    best_val_ROC_AUC = -1.0
    emb_train_loss_log = []
    clas_train_loss_log = []
    train_accuracy_log = []
    emb_val_loss_log = []
    clas_val_loss_log = []
    val_accuracy_log = []
    val_ROC_AUC_log = []
    
    for epoch in range(1, epochs + 1):
        embedding_net.train()
        classifier_net.train()
        emb_running_loss = 0.0
        clas_running_loss = 0.0
        correct_predictions = 0
        total_samples = 0

        print(f"\n==== Training Epoch {epoch} ====")

        # --- Training phase ----

        for i, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)

            # ---- Embedding model training ----
            
            embedding_opt.zero_grad()
            
            # Generate embeddings
            embeddings = embedding_net(images)
            
            # Calculate Triplet Loss using Batch-Hard mining
            emb_loss = embedding_crit(embeddings, labels)
            
            emb_loss.backward()
            embedding_opt.step()
            
            emb_running_loss += emb_loss.item() * len(images)
            
            if (i + 1) % 50 == 0:
                print(f'Epoch {epoch}/{epochs}, Batch {i+1}/{len(train_loader)}, Embedding training loss: {emb_loss.item():.4f}')

            # ---- Classification model training ----

            classifier_opt.zero_grad()
            
            # Classify
            outputs = classifier_net(embeddings)
            clas_loss = classifier_crit(outputs, labels)
            
            clas_loss.backward()
            classifier_opt.step()
            
            # Statistics
            clas_running_loss += clas_loss.item() * len(images)
            _, preds = torch.max(outputs, 1)
            correct_predictions += torch.sum(preds == labels.data).item()
            total_samples += len(images)

        # embedding training epoch loss
        emb_epoch_loss = emb_running_loss / total_samples #len(train_loader.dataset)
        print(f"Epoch {epoch} finished. \nAverage Training Embedding Loss: {emb_epoch_loss:.4f}")

        # classification training epoch loss
        clas_epoch_loss = clas_running_loss / total_samples
        epoch_acc = correct_predictions / total_samples
        print(f"Average Training Classification Loss: {clas_epoch_loss:.4f}")
        print(f"Training Classification Accuracy: {epoch_acc:.4f}")

        # ---- Evaluation phase ----

        print("--- Validation phase ---")
        val_emb_loss, val_clas_loss, epoch_val_accuracy, epoch_val_ROC_AUC = evaluate_model(embedding_net, classifier_net, embedding_crit, classifier_crit, val_loader, device)
        print(f"Average Validation Embedding Loss: {val_emb_loss:.4f}")
        print(f"Average Validation Classification Loss: {val_clas_loss:.4f}")
        print(f"Validation Classification Accuracy: {epoch_val_accuracy:.4f}")
        print(f"Validation ROC AUC: {epoch_val_ROC_AUC:.4f}")

        # metric logging for plotting
        emb_train_loss_log.append(emb_epoch_loss)
        clas_train_loss_log.append(clas_epoch_loss)
        train_accuracy_log.append(epoch_acc)
        emb_val_loss_log.append(val_emb_loss)
        clas_val_loss_log.append(val_clas_loss)
        val_accuracy_log.append(epoch_val_accuracy)
        val_ROC_AUC_log.append(epoch_val_ROC_AUC)

        scheduler.step(epoch_val_ROC_AUC)

        # save best model based on ROC AUC
        if epoch_val_ROC_AUC > best_val_ROC_AUC:
            print(f"Previous best ROC AUC: {best_val_ROC_AUC:.4f}")
            best_val_ROC_AUC = epoch_val_ROC_AUC
            # Save model checkpoint
            print("Saving best model...")
            torch.save(embedding_net.state_dict(), 'best_embedding_model.pth')
            torch.save(classifier_net.state_dict(), 'best_classifier_model.pth')


    print("Network training complete.")

    # Graphical display of metric logs
    plot_logs(
        emb_train_loss_log, 
        clas_train_loss_log,
        train_accuracy_log,
        emb_val_loss_log,
        clas_val_loss_log,
        val_accuracy_log,
        val_ROC_AUC_log,
        epochs)

In [ ]:

train_dataset = SkinDataset((data_dir / 'train'), transform=None)
val_dataset = SkinDataset((data_dir / 'val'), transform=None)

# Use standard DataLoader; Triplet mining is handled in the custom loss
train_loader = DataLoader(train_dataset, batch_size=TRAIN_BATCH_SIZE, shuffle=True, drop_last=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=VAL_TEST_BATCH_SIZE, shuffle=True, num_workers=8)

# Model Setup

# Embedding Net with Triplet Loss
embedding_net = EmbeddingNet(EMBEDDING_DIM).to(device)
triplet_criterion = TripletMarginLoss(margin=MARGIN)
embedding_optimizer = optim.Adam(embedding_net.parameters(), lr=LEARNING_RATE)
embedding_scheduler = optim.lr_scheduler.ReduceLROnPlateau(embedding_optimizer, mode='max', factor=0.5, patience=5)

# Classification Head with Cross-Entropy Loss
classifier_head = ClassificationNet(EMBEDDING_DIM).to(device)
classification_criterion = nn.CrossEntropyLoss().to(device)
classifier_optimizer = optim.Adam(classifier_head.parameters(), lr=LEARNING_RATE)# * 5) # Faster learning rate for small head

train_nets(
        embedding_net, classifier_head, 
        train_loader, val_loader, 
        triplet_criterion, classification_criterion, 
        embedding_optimizer, classifier_optimizer, 
        embedding_scheduler,
        NUM_EPOCHS, 
        device)

In [ ]:
# Evaluation
test_dataset = SkinDataset((data_dir / 'test'), transform=None)
val_loader = DataLoader(test_dataset, batch_size=VAL_TEST_BATCH_SIZE, shuffle=True, num_workers=0)
evaluate_model(embedding_net, classifier_head, val_loader, device)